In [ ]:
import numpy as np
import pandas as pd
from gensim.models import Word2Vec
from gensim.parsing.preprocessing import preprocess_string
from sklearn.model_selection import train_test_split
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.utils import pad_sequences


In [ ]:
df = pd.read_csv("IMDB Dataset.csv").dropna()
df["words"] = df.review.apply(preprocess_string)
df = df[df.words.str.len() > 0]
y = df.sentiment.map({"negative": 0, "positive": 1}).values
train, test, y_train, y_test = train_test_split(df.words.tolist(), y, test_size=0.2, stratify=y, random_state=42)


In [ ]:
cbow = Word2Vec(train, min_count=1, sg=0, epochs=20)
train = pad_sequences(train, maxlen=300, dtype=object, value="", padding="post")
test = pad_sequences(test, maxlen=300, dtype=object, value="", padding="post")
x_train = np.array([cbow.wv.get_mean_vector(w) for w in train], dtype="float32")
x_test = np.array([cbow.wv.get_mean_vector(w) for w in test], dtype="float32")


In [ ]:
model = Sequential([Input((100,)), Dense(64, "relu"), Dense(1, "sigmoid")])
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.fit(x_train, y_train, epochs=20, batch_size=128)
print("Test loss, accuracy:", model.evaluate(x_test, y_test))
